# 문제 10. 일부 분야만 검토된 인도지원 보고의 다중 라벨 분류

**개정판 v2 · 제한시간 30분 · 난이도 최상 목표 · Python3 · Jupyter · CPU · CSV 제출**

## 상황

유·무인 인도지원 상황실에서는 영문 보고를 다섯 담당 분야로 분류합니다. 학습 보고의 분야별 검토가 모두 완료되지 않아, 어떤 분야는 해당 여부가 아직 알려져 있지 않습니다. **검토된 주석을 활용하여 새 보고의 다섯 분야 해당 여부를 각각 예측하세요.** 한 보고가 여러 분야에 해당할 수 있습니다.

실제 공개 인도지원 본문과 분야 주석을 사용한 가상 MUM-T 지원 과제입니다. 일부 학습 주석을 인위적으로 숨겨 미검토 상태를 재현했습니다. 숨기지 않은 주석의 0/1은 변경하지 않았으며 원본에서 검토 누락이 발생했다는 의미는 아닙니다. 실제 군 통신 자료가 아니며 번역은 필요하지 않습니다.

## Data

| 파일 | 행 수 | 내용 |
|---|---:|---|
| data/train.csv | 7,500 | message_id, document_id, text, split, 다섯 분야 주석 |
| data/test.csv | 1,500 | message_id, document_id, text |
| data/sample_submission.csv | 1,500 | 제출 순서·형식 예시 |

- `message_id`: 본문 고유 식별자. `document_id`: 원문 문서 그룹 식별자.
- `text`: 영문 보고 발췌. `split`: fit 6,000건 / valid 1,500건.
- 같은 원문 문서·정규화한 동일 본문은 분할 간 겹치지 않습니다. 국가·프로젝트 전체를 분리한 것은 아닙니다.
- ID는 결합·검증용입니다. 입력 특징에 사용하지 마세요.

### 분야와 주석의 의미

| 정답 열 | 분야 |
|---|---|
| food | 식량 안보·식량 접근·식량 지원 |
| health | 보건·의료 |
| shelter | 주거·대피 거처 |
| wash | 식수·위생·공중위생 |
| logistics | 인도지원 물류·운송 지원 |

**fit 주석:** 1은 해당 분야의 주석 있음, 0은 해당 주석 없음, **-1은 해당 분야 미검토**입니다. -1은 세 번째 예측 클래스가 아니며 0을 뜻하지 않습니다. 한 분야가 미검토여도 다른 분야의 0/1은 유효합니다. 모든 분야가 -1인 보고도 있을 수 있습니다.

숨김 처리는 본문과 정답 값과 독립적으로 이루어졌고, 분야마다 검토 비율이 다릅니다. **valid는 다섯 분야 모두 0/1이 공개**되어 있으며 테스트 채점 정답에도 미검토 값은 없습니다. 원본 주석의 부재(0)가 모든 의미상 언급의 부재를 완벽히 보장하지는 않습니다.

다섯 값은 서로 배타적이지 않습니다. 여러 분야가 동시에 1이거나 모두 0일 수 있습니다. 요청뿐 아니라 상황·피해·대응 서술도 포함됩니다.

## 초기 코드


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path(".")
TARGETS = ["food", "health", "shelter", "wash", "logistics"]
train = pd.read_csv(BASE_DIR / "data/train.csv")
test = pd.read_csv(BASE_DIR / "data/test.csv")
sample_submission = pd.read_csv(BASE_DIR / "data/sample_submission.csv")
display(train.head())
print("학습/테스트:", train.shape, test.shape)


## 과제

1. 분야별 검토율과 알려진 라벨의 양성 비율을 확인하세요. 미검토 값을 음성으로 취급하지 않도록 학습 대상을 구성하세요.
2. fit에서 텍스트 특징과 분야별 분류 모델을 학습하세요. 분야마다 사용 가능한 정답 행이 다릅니다.
3. valid에서 Macro-F1을 계산하고 분야별 오류, 클래스 불균형, 판정 임곗값 등을 검토하여 개선하세요.
4. 최종 설정으로 train 전체를 학습에 활용하고 test의 다섯 분야를 예측하세요. 이때도 기존 fit의 -1은 정답이 아닙니다.

### 검증 조건

- 제공된 fit/valid를 사용하세요. 문서별 발췌를 무작위로 섞지 마세요.
- 검증 단계의 사전·IDF·특징 선택은 fit에서 학습합니다. fit의 미검토 본문도 정답 없이 텍스트 사전 구성에 사용할 수 있습니다.
- 확률이나 결정 점수로 판정 기준을 조정할 수 있지만 최종 제출은 **정수 0/1**입니다.
- 단일 클래스 분류처럼 가장 높은 분야 하나만 선택하지 마세요.
- 원본 주소나 이름으로 외부 정답을 검색하지 마세요. CPU로 풀이하며 외부 API나 큰 언어 모델은 필요하지 않습니다.

### 평가: Macro-F1 — 높을수록 좋음

분야마다 `F1 = 2TP / (2TP + FP + FN)`을 계산하고 다섯 F1을 동일 비중으로 평균합니다. 분모가 0이면 해당 분야 F1은 0입니다. 드문 분야도 흔한 분야와 같은 비중입니다. 모든 행·분야의 TP 등을 먼저 합치는 Micro-F1과 다릅니다.

검증 예: `f1_score(y_valid, predicted_labels, average="macro", zero_division=0)`

두 배열은 `(1500,5)`이며 열 순서는 TARGETS입니다. 모두 0인 정답행의 오탐은 해당 분야 FP에 포함됩니다. 연습 점수는 `100*Macro-F1`이고 공식 합격선은 없습니다.

## 결과 저장

Notebook과 같은 폴더에 **submission.csv**를 저장하세요.

| message_id | food | health | shelter | wash | logistics |
|---|---:|---:|---:|---:|---:|
| test.csv의 첫 번째 ID | 1 | 0 | 0 | 1 | 0 |
| test.csv의 두 번째 ID | 0 | 0 | 0 | 0 | 0 |

위 값은 형식 예시입니다.

- 컬럼 순서: `message_id,food,health,shelter,wash,logistics`.
- test.csv와 같은 ID·행 순서로 **1,500행**. 예측은 정수 0/1이며 -1은 제출할 수 없습니다.
- 모든 분야 0인 행을 허용합니다. 결측값·중복 ID·추가 컬럼은 금지합니다.
- `index=False`로 저장하고 최종 코드·실행 결과가 담긴 Notebook도 저장하세요. 별도 보고서는 없습니다.


In [ ]:
# 여기에 코드를 작성하세요.


## CSV 저장 예시

In [ ]:
# predicted_labels: test 순서, TARGETS 열 순서의 (1500,5) 정수 배열
# submission = pd.DataFrame(predicted_labels, columns=TARGETS)
# submission.insert(0, "message_id", test["message_id"].to_numpy())
# submission.to_csv(BASE_DIR / "submission.csv", index=False, encoding="utf-8-sig")


## 파일 형식 확인

In [ ]:
path = BASE_DIR / "submission.csv"
if path.exists():
    result = pd.read_csv(path)
    assert result.columns.tolist() == ["message_id"] + TARGETS
    assert result["message_id"].tolist() == test["message_id"].tolist()
    assert result["message_id"].is_unique
    assert all(pd.api.types.is_integer_dtype(result[t]) for t in TARGETS)
    assert result[TARGETS].isin([0,1]).all().all()
    print("제출 형식 확인 완료. 점수 채점 또는 최종 제출은 아닙니다.")
else:
    print("풀이 후 submission.csv를 생성하세요.")
